# Create Germany-Only Dataset

## Purpose

This notebook extracts the Germany-related columns from the full European energy dataset and saves the filtered result as a clean CSV file in the landing volume. The Germany-focused file is easier to work with in downstream processing and helps reduce storage and compute costs when building Bronze, Silver, and Gold layers.

## Steps

1. Load the project configuration (catalog names, paths, and column lists)
2. Read the full source CSV from the landing volume
3. Select only the columns defined in `germany_source_columns`
4. Write the filtered dataset to a temporary CSV folder
5. Rename the generated part file to a clean final name and remove temporary files

## Output

The final file is saved to the landing volume at the path stored in `germany_selected_file`.

In [0]:
%run ../00_setup/03-project-config 

In [0]:
output_folder_path = f"{landing_volume_path}/germany_energy_selected_columns_temp"

### Read the source file

This step loads the original energy dataset from the landing volume into a Spark DataFrame. The file is read as CSV with headers, and schema inference is turned off so the raw structure stays unchanged. This keeps the source data as close as possible to the original file before selecting only the Germany-related fields.

In [0]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(original_source_file)
)

### Select the required Germany columns

The full source file contains many columns for different countries and measurements. In this step, the notebook keeps only the columns listed in `germany_source_columns`. This creates a smaller DataFrame focused on the Germany dataset that will be used later in the lakehouse. The code also checks the row count, the number of selected columns, and shows a sample so you can confirm the result looks correct.

In [0]:
germany_df = raw_df.select(*germany_source_columns)

print("rows in germany-only file:", germany_df.count())
print("columns in germany-only file:", len(germany_df.columns))

display(germany_df.limit(10))

### Write the Germany-only dataset to a temporary CSV folder

This step writes the filtered Germany dataset back to the landing volume as a CSV file. The write uses `overwrite` mode so the temporary output can be recreated if the notebook is run again. It also uses `header = True` so the column names are included in the file.

Spark writes CSV output into a folder, not directly into a single named file. `coalesce(1)` is used here to create only one CSV part file, which makes it easier to rename in the next step. The landing layer remains file-based, while the conversion to Delta happens later during Bronze ingestion.

In [0]:
(
    germany_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_folder_path)
)

In [0]:
display(dbutils.fs.ls(landing_volume_path))

### Rename the generated file and clean up the temporary folder

After Spark writes the output, the actual CSV is stored inside the temporary folder with a system-generated `part-...csv` name. This step finds that file, moves it to the final target path stored in `germany_selected_file`, and removes the temporary folder.

At the end, the notebook lists the landing volume again so you can confirm that the clean final file was created successfully.

In [0]:

part_file = [
    file.path
    for file in dbutils.fs.ls(output_folder_path)
    if file.name.startswith("part-") and file.name.endswith(".csv")
][0]

dbutils.fs.rm(germany_selected_file, recurse=True)
dbutils.fs.mv(part_file, germany_selected_file)
dbutils.fs.rm(output_folder_path, recurse=True)

display(dbutils.fs.ls(landing_volume_path))